# reglscatterpy — full feature tour

The Python companion to **reglScatterplotR**. By default it renders the *same* compiled widget the R package ships (legend, lasso, tooltips, sync, PNG/SVG/PDF export) via [anywidget](https://anywidget.dev), so plots are pixel-identical across R and Python and work in **Jupyter, JupyterLab, VS Code and Colab**.

```bash
pip install -e .            # from the python/ dir; pulls numpy, pandas, anywidget
pip install anndata         # for the AnnData examples
```

Run cells top-to-bottom. Each `scatterplot(...)` call displays an interactive widget.

> **Sizing.** Plots fill the cell width by default; pass `width=` (pixels) for a fixed size, `height=` for the height.


## 0. Setup

In [1]:
import numpy as np
import pandas as pd
import reglscatterpy as rs
print('reglscatterpy', rs.__version__)

reglscatterpy 0.2.0


A small synthetic single-cell-style DataFrame (mirrors the R `reglScatterExample`).

In [2]:
rng = np.random.default_rng(0)
n = 3000
ct = rng.choice(['CD4 T','CD8 T','NK','B','Monocyte','Dendritic'], size=n)
centers = {k:(rng.uniform(-6,6), rng.uniform(-6,6)) for k in set(ct)}
umap1 = np.array([centers[c][0] for c in ct]) + rng.normal(0,1,n)
umap2 = np.array([centers[c][1] for c in ct]) + rng.normal(0,1,n)
df = pd.DataFrame({'UMAP_1':umap1, 'UMAP_2':umap2, 'celltype':ct,
                   'nCount':rng.integers(2000,12000,n), 'CD3D':rng.gamma(2,1,n)})
df.head()

,UMAP_1,UMAP_2,celltype,nCount,CD3D
0,4.609517,2.853602,Dendritic,4254,1.237779
1,3.983760,-4.789348,B,3471,1.184316
2,4.263779,-4.349272,B,9525,3.393274
3,-4.743711,0.662437,CD8 T,2782,1.742883
4,-7.173700,0.561905,CD8 T,6909,1.757534


## 1. Basics

### 1.1 Minimal plot (DataFrame)

In [7]:
rs.scatterplot(df, x='UMAP_1', y='UMAP_2')

<reglscatterpy._widget._make_class.<locals>.ReglScatter object at 0x7f24a4e3bd90>

### 1.2 Colour by a categorical column

In [8]:
rs.scatterplot(df, x='UMAP_1', y='UMAP_2', color_by='celltype', legend_title='Cell type')

<reglscatterpy._widget._make_class.<locals>.ReglScatter object at 0x7f24a4e3b250>

### 1.3 Colour by a continuous column

In [9]:
rs.scatterplot(df, x='UMAP_1', y='UMAP_2', color_by='CD3D', continuous_palette='viridis', legend_title='CD3D')

<reglscatterpy._widget._make_class.<locals>.ReglScatter object at 0x7f24a4e19bd0>

### 1.4 Fixed point colour

In [10]:
rs.scatterplot(df, x='UMAP_1', y='UMAP_2', point_color='#0072B2')

<reglscatterpy._widget._make_class.<locals>.ReglScatter object at 0x7f24e421cc50>

## 2. Point appearance

### 2.1 Size & opacity

In [11]:
rs.scatterplot(df, x='UMAP_1', y='UMAP_2', color_by='celltype', point_size=6, opacity=0.5)

<reglscatterpy._widget._make_class.<locals>.ReglScatter object at 0x7f24a4e34d50>

### 2.2 `pixel_ratio` (crisp / export-quality)

In [13]:
rs.scatterplot(df, x='UMAP_1', y='UMAP_2', color_by='celltype', pixel_ratio=10)

<reglscatterpy._widget._make_class.<locals>.ReglScatter object at 0x7f24d5726f50>

## 3. Palettes

Continuous (viridis family) and categorical (ColorBrewer) palettes are byte-identical to the R package.

In [14]:
rs.scatterplot(df, x='UMAP_1', y='UMAP_2', color_by='nCount', continuous_palette='magma', legend_title='nCount')

<reglscatterpy._widget._make_class.<locals>.ReglScatter object at 0x7f24a67cc610>

### 3.1 Custom per-category colours

In [15]:
my = {c:h for c,h in zip(sorted(set(ct)), ['#e6194B','#3cb44b','#4363d8','#f58231','#911eb4','#42d4f4'])}
rs.scatterplot(df, x='UMAP_1', y='UMAP_2', color_by='celltype', custom_colors=my)

<reglscatterpy._widget._make_class.<locals>.ReglScatter object at 0x7f24a4e57650>

## 4. Continuous scaling: `vmin` / `vmax` / `center_zero`

In [16]:
rs.scatterplot(df, x='UMAP_1', y='UMAP_2', color_by='CD3D', vmin=0, vmax='p95')

<reglscatterpy._widget._make_class.<locals>.ReglScatter object at 0x7f24d4536dd0>

## 5. Filtering

`filter_by` adds interactive numeric range filters.

In [5]:
rs.scatterplot(df, x='UMAP_1', y='UMAP_2', color_by='celltype', filter_by=df[['nCount','CD3D']])

<reglscatterpy._widget._make_class.<locals>.ReglScatter object at 0x7f023c05e7d0>

## 6. Titles, axes, theming & legend

In [18]:
rs.scatterplot(df, x='UMAP_1', y='UMAP_2', color_by='celltype',
    title='PBMC — UMAP', xlab='UMAP 1', ylab='UMAP 2',
    background_color='#111418', axis_color='#cccccc',
    legend_bg='#1c2128', legend_text='#e6edf3', legend_position='bottom-left')

<reglscatterpy._widget._make_class.<locals>.ReglScatter object at 0x7f020bab01d0>

## 7. Export (PNG / SVG / PDF)

In [4]:
rs.scatterplot(df, x='UMAP_1', y='UMAP_2', color_by='celltype', enable_download=True)

<reglscatterpy._widget._make_class.<locals>.ReglScatter object at 0x7f023c08ae50>

## 8. AnnData / scanpy

Coordinates from `obsm`, colour from an `obs` column or a gene in `var_names`.

In [19]:
try:
    import anndata as ad
    g = 30
    X = rng.poisson(2, size=(n, g)).astype(float)
    adata = ad.AnnData(X=X,
        obs=pd.DataFrame({'celltype': pd.Categorical(ct)}),
        var=pd.DataFrame(index=[f'Gene{i}' for i in range(g)]))
    adata.obsm['X_umap'] = np.c_[umap1, umap2]
    display(rs.scatterplot(adata, x='umap', color_by='celltype'))   # obsm + obs column
except ModuleNotFoundError:
    print('anndata not installed — pip install anndata')

/home/goguxor/miniconda3/envs/reglpy/lib/python3.11/functools.py:909: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


<reglscatterpy._widget._make_class.<locals>.ReglScatter object at 0x7f24a1a38190>

### 8.1 Colour by a gene (read from `.X`)

In [21]:
try:
    import anndata  # noqa
    display(rs.scatterplot(adata, x='umap', color_by='Gene0', continuous_palette='viridis'))
except (ModuleNotFoundError, NameError):
    print('anndata not available')

<reglscatterpy._widget._make_class.<locals>.ReglScatter object at 0x7f24a4b6afd0>

## 9. numpy array input

In [22]:
emb = np.c_[umap1, umap2]
rs.scatterplot(emb, color_by=ct)        # first two columns are x/y

<reglscatterpy._widget._make_class.<locals>.ReglScatter object at 0x7f24a4e63250>

## 10. MuData (multimodal) and SpatialData

For multimodal data, address a feature as `"modality:feature"` and an embedding as `"modality:embedding"` (e.g. `"rna:X_umap"`).

In [6]:
try:
    import anndata as ad, mudata as mu
    rng = np.random.default_rng(0); m = 800
    cells_idx = [f'c{i}' for i in range(m)]
    ctm = rng.choice(['T','B','NK'], m)
    rna = ad.AnnData(rng.poisson(2,(m,20)).astype(float),
        obs=pd.DataFrame({'celltype': pd.Categorical(ctm)}, index=cells_idx),
        var=pd.DataFrame(index=[f'Gene{i}' for i in range(20)]))
    rna.obsm['X_umap'] = rng.normal(0,1,(m,2))
    adt = ad.AnnData(rng.poisson(5,(m,8)).astype(float),
        obs=pd.DataFrame(index=cells_idx),
        var=pd.DataFrame(index=[f'AB{i}' for i in range(8)]))
    mdata = mu.MuData({'rna': rna, 'adt': adt})
    display(rs.scatterplot(mdata, x='rna:X_umap', color_by='rna:celltype'))   # obs of a modality
except ModuleNotFoundError:
    print('mudata not installed — pip install mudata')

/home/goguxor/miniconda3/envs/reglpy/lib/python3.11/site-packages/mudata/_core/mudata.py:1416: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)
/home/goguxor/miniconda3/envs/reglpy/lib/python3.11/site-packages/mudata/_core/mudata.py:1272: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("obs", axis=1, join_common=join_common)


<reglscatterpy._widget._make_class.<locals>.ReglScatter object at 0x7f020bae3550>

### 10.1 Colour a multimodal embedding by a feature from another modality

In [7]:
try:
    import mudata  # noqa
    display(rs.scatterplot(mdata, x='rna:X_umap', color_by='adt:AB0'))   # a protein (ADT) feature
except (ModuleNotFoundError, NameError):
    print('mudata not available')

<reglscatterpy._widget._make_class.<locals>.ReglScatter object at 0x7f020c1f2250>

### 10.2 SpatialData

A `SpatialData` object's annotation table is used; `x='spatial'` (the default for SpatialData) reads `obsm['spatial']`. Pass `table=` to pick a specific table.

In [8]:
try:
    import spatialdata as sd  # heavy optional dep
    # rs.scatterplot(sdata, x='spatial', color_by='region')
    # rs.scatterplot(sdata, table='table_name', x='spatial', color_by='celltype')
    print('spatialdata available — use the calls above with your object')
except ModuleNotFoundError:
    print('spatialdata not installed — pip install spatialdata (heavy). '
          'Usage: rs.scatterplot(sdata, x="spatial", color_by="region")')

spatialdata not installed — pip install spatialdata (heavy). Usage: rs.scatterplot(sdata, x="spatial", color_by="region")


## 11. Note on backends

Everything above uses the default native widget. A `backend="jscatter"` option also exists if you prefer [jupyter-scatter](https://github.com/flekschas/jupyter-scatter) (`pip install reglscatterpy[render]`) — but the default is recommended.


## 12. Inspecting the extracted data

In [24]:
from reglscatterpy import extract
pd_data = extract(df, x='UMAP_1', y='UMAP_2', color_by='celltype')
print('n =', pd_data.n, '| color_name =', pd_data.color_name)
pd_data.x[:5], pd_data.y[:5]

n = 3000 | color_name = celltype


(array([ 3.26565022, -5.32292605, -5.04290726, -1.48836223, -3.91835119]),
 array([-4.05940634, -5.00353729, -4.56346163, -2.66346305, -2.76399478]))